In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# ===============================
# CONFIGURATION
# ===============================

SERVICE_LEVEL = 0.95
Z = norm.ppf(SERVICE_LEVEL)

# ===============================
# LOAD FILE
# ===============================

df = pd.read_excel("indent_vs_actual_wide_analysis.xlsx")

# ===============================
# IDENTIFY DAILY INDENT COLUMNS ONLY
# ===============================

indent_cols = [
    col for col in df.columns
    if "Indent" in col
    and "Total" not in col
    and "Deviation" not in col
]

print("Indent columns used:")
print(indent_cols)

# ===============================
# CALCULATE MEAN & STD ON INDENT
# ===============================

df["Mean_Indent"] = df[indent_cols].mean(axis=1)
df["Std_Indent"] = df[indent_cols].std(axis=1)

df["Std_Indent"] = df["Std_Indent"].fillna(0)

# ===============================
# CONFIDENCE INTERVAL
# ===============================

df["Indent_CI_Lower"] = df["Mean_Indent"] - Z * df["Std_Indent"]
df["Indent_CI_Upper"] = df["Mean_Indent"] + Z * df["Std_Indent"]

df["Indent_CI_Lower"] = df["Indent_CI_Lower"].apply(lambda x: max(0, x))

# ===============================
# SAVE
# ===============================

df.to_excel("indent_confidence_band_clean.xlsx", index=False)

print("Indent confidence band calculated correctly.")


In [ ]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

# Load indent confidence interval file
ci_df = pd.read_excel("indent_confidence_band_clean.xlsx")

# Load updated actual file
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# MERGE 12th & 13th ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH FEB AGAINST INDENT CI
# =====================================

df["12th_Inside_Indent_CI"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper"])
)

# =====================================
# VALIDATE 13TH FEB AGAINST INDENT CI
# =====================================

df["13th_Inside_Indent_CI"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper"])
)

# =====================================
# HIT RATE CALCULATION
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI"].mean() * 100

print("12th Feb Hit Rate (Indent CI):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent CI):", round(hit_rate_13, 2), "%")

# Save validation file
df.to_excel("Indent_CI_validation_12_13.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD INDENT CI FILE
# =====================================

df = pd.read_excel("indent_confidence_band_clean.xlsx")

# =====================================
# RECOMPUTE INDENT CI USING ±2σ
# =====================================

Z = 2  # ±2 sigma

df["Indent_CI_Lower_2sigma"] = df["Mean_Indent"] - Z * df["Std_Indent"]
df["Indent_CI_Upper_2sigma"] = df["Mean_Indent"] + Z * df["Std_Indent"]

df["Indent_CI_Lower_2sigma"] = df["Indent_CI_Lower_2sigma"].clip(lower=0)

# =====================================
# VALIDATE 12TH FEB
# =====================================

df["12th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH FEB
# =====================================

df["13th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# HIT RATE
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (Indent ±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent ±2σ):", round(hit_rate_13, 2), "%")

# Save
df.to_excel("Indent_CI_2sigma_validation.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

ci_df = pd.read_excel("confidence_band_analysis.xlsx")
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# RECOMPUTE CI USING ±2σ
# =====================================

ci_df["CI_Lower_2sigma"] = ci_df["Mean_Actual"] - 2 * ci_df["Std_Actual"]
ci_df["CI_Upper_2sigma"] = ci_df["Mean_Actual"] + 2 * ci_df["Std_Actual"]

ci_df["CI_Lower_2sigma"] = ci_df["CI_Lower_2sigma"].clip(lower=0)

# =====================================
# MERGE 12th & 13th ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH FEB (±2σ)
# =====================================

df["12th_Inside_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH FEB (±2σ)
# =====================================

df["13th_Inside_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

# =====================================
# HIT RATE CALCULATION
# =====================================

hit_rate_12 = df["12th_Inside_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (±2σ):", round(hit_rate_13, 2), "%")

# Save validation file
df.to_excel("CI_validation_12_13_2sigma.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD INDENT CI FILE
# =====================================

ci_df = pd.read_excel("indent_confidence_band_clean.xlsx")
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# COMPUTE ±2σ ON INDENT
# =====================================

ci_df["Indent_CI_Lower_2sigma"] = ci_df["Mean_Indent"] - 2 * ci_df["Std_Indent"]
ci_df["Indent_CI_Upper_2sigma"] = ci_df["Mean_Indent"] + 2 * ci_df["Std_Indent"]

ci_df["Indent_CI_Lower_2sigma"] = ci_df["Indent_CI_Lower_2sigma"].clip(lower=0)

# =====================================
# MERGE 12TH & 13TH ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH (Indent ±2σ)
# =====================================

df["12th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH (Indent ±2σ)
# =====================================

df["13th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# HIT RATE
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (Indent ±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent ±2σ):", round(hit_rate_13, 2), "%")

# Save
df.to_excel("Indent_CI_2sigma_validation.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np

# =====================================
# 1. LOAD FILES (same as your script)
# =====================================
ci_df    = pd.read_excel("confidence_band_analysis.xlsx")
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# 2. RECOMPUTE ±2σ BANDS (same names as your code)
# =====================================
ci_df["CI_Lower_2sigma"] = ci_df["Mean_Actual"] - 2 * ci_df["Std_Actual"]
ci_df["CI_Upper_2sigma"] = ci_df["Mean_Actual"] + 2 * ci_df["Std_Actual"]
ci_df["CI_Lower_2sigma"] = ci_df["CI_Lower_2sigma"].clip(lower=0)

# =====================================
# 3. MERGE 12th & 13th PLAN DATA (same columns)
# =====================================
cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]
actual_subset = actual_df[cols_to_merge]
df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# 4. LOGIC SIMULATION & COMPARISON
# =====================================
results = []

for idx, row in df.iterrows():
    material = row["Material"]
    mu       = row["Mean_Actual"]
    lower    = row["CI_Lower_2sigma"]
    upper    = row["CI_Upper_2sigma"]
    
    if mu <= 0:
        continue
    
    # Start with zero inventory
    inventory = 0.0
    
    for date in ["2026-02-12", "2026-02-13"]:
        plan_col = f"{date} Total Production Plan"
        D = row[plan_col] if pd.notna(row[plan_col]) else mu   # forecast or fallback to mean
        
        # ── Your buffer rule ──
        days_cover = inventory / mu if mu > 0 else 0
        if days_cover >= 3.0:
            target_buffer_days = 0
        else:
            target_buffer_days = int(np.floor(days_cover)) + 1   # 0.xx→1, 1.xx→2, 2.xx→3
        
        target_inventory = target_buffer_days * mu
        extra_needed     = max(0.0, target_inventory - inventory)
        wish_quantity    = D + extra_needed
        
        # ── Clamp to ±2σ band (this is what keeps the line stable) ──
        recommended_qty = max(lower, min(upper, wish_quantity))
        recommended_qty = round(recommended_qty, 0)
        
        # Reason (for easy explanation)
        reasons = []
        if target_buffer_days > 0:
            reasons.append(f"Build {target_buffer_days}d buffer")
        if recommended_qty > D:
            reasons.append(f"+{recommended_qty - D:.0f} extra")
        elif recommended_qty < D:
            reasons.append("Draw from stock")
        if recommended_qty >= upper:
            reasons.append("Upper cap")
        elif recommended_qty <= lower:
            reasons.append("Lower cap")
        reason_str = " | ".join(reasons) or "Steady"
        
        # Projected end-of-day inventory using RECOMMENDED qty
        end_inv = max(0.0, inventory + recommended_qty - D)
        
        # Differences vs actual plan
        actual_plan_qty = D   # since D is taken from the plan column
        diff_units   = recommended_qty - actual_plan_qty
        diff_pct     = (diff_units / actual_plan_qty * 100) if actual_plan_qty > 0 else 0
        
        results.append({
            "Material": material,
            "Date": date,
            "Actual_Plan_Qty": actual_plan_qty,
            "Logic_Recommended_Qty": recommended_qty,
            "Diff_Units": diff_units,
            "Diff_%": round(diff_pct, 2),
            "Days_Cover_Start": round(days_cover, 2),
            "Target_Buffer_Days": target_buffer_days,
            "End_Inventory_Logic": round(end_inv, 1),
            "Reason": reason_str,
            "Plan_Was_Inside_Band": (lower <= actual_plan_qty <= upper)
        })
        
        # Carry forward the LOGIC's inventory (not the actual plan's)
        inventory = end_inv

# =====================================
# 5. CREATE REPORT DATAFRAMES
# =====================================
report_df = pd.DataFrame(results)

# Summary statistics
n_materials = df["Material"].nunique()
hit_rate_12 = df["12th_Inside_CI_2sigma"].mean() * 100 if "12th_Inside_CI_2sigma" in df.columns else np.nan
hit_rate_13 = df["13th_Inside_CI_2sigma"].mean() * 100 if "13th_Inside_CI_2sigma" in df.columns else np.nan

summary = {
    "Materials": n_materials,
    "Original hit rate 12th (%)": round(hit_rate_12, 2),
    "Original hit rate 13th (%)": round(hit_rate_13, 2),
    "Avg |Diff| (units)": round(report_df["Diff_Units"].abs().mean(), 1),
    "Avg Diff %": round(report_df["Diff_%"].abs().mean(), 2),
    "Cases where logic differs >10%": (report_df["Diff_%"].abs() > 10).sum(),
    "Avg end inventory 13th (logic)": round(
        report_df[report_df["Date"] == "2026-02-13"]["End_Inventory_Logic"].mean(), 1
    )
}

print("\n" + "="*70)
print("COMPARISON: Your actual plans vs Proposed stable logic (inventory starts at 0)")
print("="*70)
for k, v in summary.items():
    print(f"{k:38} : {v}")
print("="*70)

# Save results
report_df.to_excel("Logic_vs_ActualPlan_12_13_Feb_Detailed.xlsx", index=False)

# Optional: wide format for quick view
pivot = report_df.pivot(
    index="Material",
    columns="Date",
    values=["Logic_Recommended_Qty", "Actual_Plan_Qty", "Diff_%", "Reason"]
).reset_index()
pivot.to_excel("Logic_vs_ActualPlan_12_13_Feb_Pivot.xlsx", index=False)

print("\nFiles created:")
print("• Logic_vs_ActualPlan_12_13_Feb_Detailed.xlsx   → full day-by-day comparison")
print("• Logic_vs_ActualPlan_12_13_Feb_Pivot.xlsx      → one row per material")